# Simulador 13.2 — Reforzamiento Diferencial de Tiempos Entre Respuestas

**Capítulo 13: La Ley del Efecto y los Programas de Refuerzo**  
*Aprendizaje y Comportamiento Adaptable: Principios y Modelos*  
Arturo Bouzas · Laboratorio 25 · UNAM

---

Los organismos responden en **ráfagas**: series rápidas de respuestas separadas por pausas.
La distribución de tiempos entre respuestas (TER) es bimodal: un modo en TER cortos (dentro de
la ráfaga) y uno en TER largos (entre ráfagas).

Este simulador muestra que los dos programas seleccionan partes distintas de esa distribución:

- **RV**: el reforzador se entrega después de completar un número de respuestas → es más probable
  que caiga *en medio de una ráfaga* → **refuerza TERs cortos**.
- **IV**: el reforzador se entrega a la primera respuesta tras el intervalo → esa respuesta
  es típicamente la primera de una nueva ráfaga, que sigue a una pausa → **refuerza TERs largos**.

A lo largo del entrenamiento, esta selección diferencial modifica la distribución de TERs
del organismo: bajo RV las ráfagas se prolongan; bajo IV las pausas se alargan.

> **Instrucción:** Ejecuta todas las celdas (*Runtime → Run all*). Ajusta los sliders y presiona
> **▶ Simular**. Registra tu predicción *antes* de cambiar los parámetros.


In [ ]:
# En Google Colab, ipywidgets suele estar disponible.
try:
    import ipywidgets
    print("✓ ipywidgets", ipywidgets.__version__)
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "ipywidgets", "--quiet"])
    print("ipywidgets instalado — reinicia el runtime y vuelve a ejecutar.")


In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
import ipywidgets as widgets
from IPython.display import display, clear_output

matplotlib.rcParams.update({
    'font.family':      'serif',
    'font.serif':       ['Georgia', 'Palatino Linotype', 'DejaVu Serif'],
    'axes.spines.top':  False,
    'axes.spines.right': False,
})

# ── Paleta del libro ──────────────────────────────────────────────────────
AZUL     = '#2C5282'
NARANJA  = '#C05621'
VERDE    = '#276749'
GRIS     = '#718096'
GRIS_MED = '#A0AEC0'
GRIS_CL  = '#EDF2F7'
BLANCO   = '#FFFFFF'
COLOR_RV = AZUL
COLOR_IV = NARANJA

print("✓ Importaciones y paleta configuradas.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# FUNCIONES DE SIMULACIÓN
# ─────────────────────────────────────────────────────────────────────────

def generar_irts_rafaga(n_irts, irt_corto, irt_largo, tam_rafaga, seed):
    """
    Genera n_irts TERs con estructura explícita de ráfaga:
      · Dentro de ráfaga: TER ~ Exp(irt_corto)
      · Primera respuesta de nueva ráfaga: TER ~ Exp(irt_largo)
      · Tamaño de ráfaga: Geométrica(1/tam_rafaga)

    Devuelve:
        irts     : array de n_irts TERs
        tiempos  : array de n_irts tiempos acumulados (tiempo de cada respuesta)
        es_corto : bool array, True si el TER es within-burst (corto)
    """
    rng = np.random.default_rng(seed)
    irts     = np.empty(n_irts)
    es_corto = np.zeros(n_irts, dtype=bool)

    i = 0
    t = 0.0
    while i < n_irts:
        # ── Nueva ráfaga: el primer TER es LARGO (pausa entre ráfagas) ──
        if i > 0:                          # no hay TER antes de la primera respuesta
            irt = rng.exponential(irt_largo)
            irts[i]     = irt
            es_corto[i] = False
            t += irt
            i += 1
            if i >= n_irts:
                break

        # ── Tamaño de esta ráfaga ─────────────────────────────────────
        tam = max(1, int(rng.geometric(1.0 / tam_rafaga)))

        # ── Respuestas dentro de la ráfaga: TER CORTO ─────────────────
        for _ in range(tam - 1):
            if i >= n_irts:
                break
            irt = rng.exponential(irt_corto)
            irts[i]     = irt
            es_corto[i] = True
            t += irt
            i += 1

    tiempos = np.cumsum(irts)
    return irts, tiempos, es_corto


def encontrar_reforzados_rv(irts, rv_n, seed):
    """
    RV-n: reforzar cada ~rv_n respuestas (requisito ~ Geométrica con media rv_n).
    Devuelve los TERs que precedieron a una respuesta reforzada.
    """
    rng = np.random.default_rng(seed)
    ref = []
    req   = max(1, int(rng.geometric(1.0 / rv_n)))
    count = 0
    for k in range(len(irts)):
        count += 1
        if count >= req:
            ref.append(irts[k])
            count = 0
            req = max(1, int(rng.geometric(1.0 / rv_n)))
    return np.array(ref) if ref else np.array([1e-3])


def encontrar_reforzados_iv(irts, tiempos, iv_t, seed):
    """
    IV-t: reforzar la primera respuesta tras un intervalo ~Exp(iv_t).
    Devuelve los TERs que precedieron a una respuesta reforzada.
    """
    rng = np.random.default_rng(seed)
    ref    = []
    t_prox = rng.exponential(iv_t)
    for k in range(len(irts)):
        if tiempos[k] >= t_prox:
            ref.append(irts[k])
            t_prox = tiempos[k] + rng.exponential(iv_t)
    return np.array(ref) if ref else np.array([1e-3])


def prob_relativa_reforzamiento(irts_all, irts_ref, bins):
    """
    Probabilidad relativa de reforzamiento como función del TER:
        P(reforzado | TER ∈ bin) / P(reforzado)
    Valores > 1: sobre-representación; < 1: sub-representación.
    Una curva plana en 1.0 indicaría selección proporcional a la base.
    """
    n_all, _ = np.histogram(irts_all, bins=bins)
    n_ref, _ = np.histogram(irts_ref, bins=bins)
    tasa_base = len(irts_ref) / len(irts_all)
    with np.errstate(divide='ignore', invalid='ignore'):
        prob = np.where(n_all > 5,
                        (n_ref / n_all) / tasa_base,
                        np.nan)
    centros = np.sqrt(bins[:-1] * bins[1:])   # media geométrica del bin
    return centros, prob


def simular_evolucion(irt_corto, irt_largo, tam_rafaga,
                       rv_n, iv_t, alpha, n_epochs,
                       n_por_epoch=1200, seed_base=42):
    """
    Simula n_epochs bloques de entrenamiento.
    Parámetro adaptativo: tam_rafaga (tamaño medio de ráfaga).
      · RV refuerza TERs cortos → tam_rafaga crece → más respuestas por ráfaga
      · IV refuerza TERs largos → tam_rafaga decrece → ráfagas más cortas
    (La pausa entre ráfagas irt_largo se mantiene fija para aislar el efecto.)

    Devuelve:
        historial con p_corto (proporción de TERs cortos) por bloque,
        y snapshots de la distribución de TERs en bloques seleccionados.
    """
    tam_rv = float(tam_rafaga)
    tam_iv = float(tam_rafaga)
    threshold = np.sqrt(irt_corto * irt_largo)

    def p_corto_de_tam(tam):
        "Proporción teórica de TERs cortos dado el tamaño medio de ráfaga."
        return (tam - 1) / tam if tam > 1 else 0.0

    hist = {
        'p_rv': [p_corto_de_tam(tam_rv)],
        'p_iv': [p_corto_de_tam(tam_iv)],
        'snaps_rv': {},
        'snaps_iv': {},
    }
    snap_set = {0, n_epochs // 3, 2 * n_epochs // 3, n_epochs - 1}

    for ep in range(n_epochs):
        s = seed_base + ep

        irts_rv, t_rv, _ = generar_irts_rafaga(n_por_epoch, irt_corto, irt_largo,
                                                tam_rv, s)
        irts_iv, t_iv, _ = generar_irts_rafaga(n_por_epoch, irt_corto, irt_largo,
                                                tam_iv, s + 5000)

        ref_rv = encontrar_reforzados_rv(irts_rv, rv_n, s + 10000)
        ref_iv = encontrar_reforzados_iv(irts_iv, t_iv, iv_t, s + 20000)

        if ep in snap_set:
            hist['snaps_rv'][ep] = irts_rv.copy()
            hist['snaps_iv'][ep] = irts_iv.copy()

        # Fracción de TERs reforzados que son CORTOS
        psr = float(np.mean(ref_rv < threshold))
        psi = float(np.mean(ref_iv < threshold))

        # El tamaño de ráfaga se ajusta proporcionalmente:
        # alta psr (refuerza cortos) → ráfagas más largas (más TERs cortos)
        # baja psi (refuerza largos) → ráfagas más cortas (más TERs largos)
        tam_rv = np.clip((1 - alpha) * tam_rv + alpha * (1 + psr * (tam_rafaga * 2 - 1)),
                         1.5, tam_rafaga * 5)
        tam_iv = np.clip((1 - alpha) * tam_iv + alpha * (1 + psi  * (tam_rafaga * 2 - 1)),
                         1.5, tam_rafaga * 5)

        hist['p_rv'].append(p_corto_de_tam(tam_rv))
        hist['p_iv'].append(p_corto_de_tam(tam_iv))

    hist['threshold'] = threshold
    return hist

print("✓ Funciones de simulación definidas.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# FUNCIONES DE GRAFICACIÓN
# ─────────────────────────────────────────────────────────────────────────

def _estilo_ax(ax):
    ax.tick_params(colors=GRIS, labelsize=8)
    ax.grid(True, alpha=0.14, color=GRIS_MED)
    ax.set_facecolor(BLANCO)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRIS_CL)


def _bins(irt_corto, irt_largo, n=50):
    lo = max(irt_corto * 0.05, 0.01)
    hi = irt_largo * 30
    return np.geomspace(lo, hi, n + 1)


def _panel_histograma(ax, irts_all, irts_ref, color, titulo,
                       threshold, irt_corto, irt_largo, bins):
    """
    Panel superior: todos los TERs (gris) + TERs reforzados (color).
    Escala log en el eje X para visualizar la distribución bimodal.
    """
    ax.hist(np.clip(irts_all, bins[0], bins[-1]),
            bins=bins, density=True, color=GRIS, alpha=0.40,
            label='Todos los TERs')
    ax.hist(np.clip(irts_ref, bins[0], bins[-1]),
            bins=bins, density=True, color=color, alpha=0.80,
            label='TERs reforzados')

    # Líneas de referencia
    ax.axvline(threshold,  color='#2d2d2d', linewidth=1.3, linestyle='--', alpha=0.6,
               label=f'umbral ({threshold:.2f} s)')
    ax.axvline(irt_corto,  color=color,      linewidth=0.9, linestyle=':', alpha=0.55)
    ax.axvline(irt_largo,  color=GRIS,       linewidth=0.9, linestyle=':', alpha=0.55)

    # Etiquetas de los modos
    ymax = ax.get_ylim()[1]
    ax.text(irt_corto * 1.15, ymax * 0.92,
            f'TER corto\n(μ={irt_corto:.2f}s)',
            fontsize=7.5, color=color, alpha=0.8, ha='left')
    ax.text(irt_largo * 1.15, ymax * 0.92,
            f'TER largo\n(μ={irt_largo:.1f}s)',
            fontsize=7.5, color=GRIS, alpha=0.8, ha='left')

    # Estadísticas
    pct_c = 100.0 * np.mean(irts_ref < threshold)
    ax.text(0.97, 0.06,
            f'Reforzados cortos: {pct_c:.0f}%\nReforzados largos: {100-pct_c:.0f}%',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=9, color=color, fontweight='bold',
            bbox=dict(facecolor=BLANCO, edgecolor=color,
                      boxstyle='round,pad=0.35', alpha=0.93))

    ax.set_xscale('log')
    ax.set_xlabel('TER (s, escala log)', fontsize=9, color=GRIS)
    ax.set_ylabel('Densidad relativa', fontsize=9, color=GRIS)
    ax.set_title(titulo, fontsize=11, fontweight='bold', color=color, pad=5)
    ax.legend(fontsize=7.5, framealpha=0.9, edgecolor=GRIS_MED)
    _estilo_ax(ax)


def _panel_prob_cond(ax, centros_rv, prob_rv, centros_iv, prob_iv, threshold):
    """
    Panel medio: probabilidad RELATIVA de reforzamiento en función del TER.
    Valor 1 = tasa de reforzamiento proporcional a la frecuencia base.
    RV ≈ línea plana; IV sube para TERs largos.
    """
    # Eliminar NaNs para la gráfica
    m_rv = ~np.isnan(prob_rv)
    m_iv = ~np.isnan(prob_iv)

    ax.plot(centros_rv[m_rv], prob_rv[m_rv],
            color=COLOR_RV, linewidth=2.2, label='RV', zorder=4)
    ax.plot(centros_iv[m_iv], prob_iv[m_iv],
            color=COLOR_IV, linewidth=2.2, linestyle='--', label='IV', zorder=4)

    ax.axhline(1.0, color='#2d2d2d', linewidth=1.1, linestyle=':', alpha=0.55,
               label='nivel de azar')
    ax.axvline(threshold, color='#2d2d2d', linewidth=1.1, linestyle='--', alpha=0.45)

    ax.fill_between(centros_rv[m_rv], 1, prob_rv[m_rv],
                    where=prob_rv[m_rv] > 1,
                    color=COLOR_RV, alpha=0.12)
    ax.fill_between(centros_iv[m_iv], 1, prob_iv[m_iv],
                    where=prob_iv[m_iv] > 1,
                    color=COLOR_IV, alpha=0.12)

    ax.set_xscale('log')
    ax.set_xlabel('TER (s, escala log)', fontsize=9, color=GRIS)
    ax.set_ylabel('P(reforzado|TER) / P(reforzado)', fontsize=9, color=GRIS)
    ax.set_title('Probabilidad relativa de reforzamiento por TER',
                 fontsize=10, color=GRIS, fontweight='bold', pad=4)
    ax.legend(fontsize=9, framealpha=0.9, edgecolor=GRIS_MED)
    ax.set_ylim(0)
    _estilo_ax(ax)

    # Anotaciones explicativas
    ax.text(0.02, 0.93,
            'RV ≈ plano (selección proporcional a la frecuencia base)',
            transform=ax.transAxes, fontsize=8, color=COLOR_RV, va='top')
    ax.text(0.02, 0.83,
            'IV ↑ para TERs largos (selección sesgada)',
            transform=ax.transAxes, fontsize=8, color=COLOR_IV, va='top')


def _panel_evolucion(ax, hist, p0, rv_n, iv_t):
    """
    Panel inferior izquierdo: proporción de TERs cortos por bloque de entrenamiento.
    """
    x = range(len(hist['p_rv']))
    ax.plot(x, hist['p_rv'], color=COLOR_RV, linewidth=2.3,
            label=f'RV-{rv_n}', solid_capstyle='round')
    ax.plot(x, hist['p_iv'], color=COLOR_IV, linewidth=2.3,
            linestyle='--', label=f'IV-{iv_t}s', solid_capstyle='round')
    ax.axhline(p0, color=GRIS, linewidth=1.1, linestyle=':', alpha=0.65,
               label=f'línea base (p₀={p0:.2f})')

    # Anotar valores finales
    xf = len(x) - 1
    p_rv_f = hist['p_rv'][-1]
    p_iv_f = hist['p_iv'][-1]
    ax.annotate(f'{p_rv_f:.2f}',
                xy=(xf, p_rv_f), xytext=(xf - max(1, xf//5), p_rv_f + 0.06),
                fontsize=9, color=COLOR_RV, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=COLOR_RV, lw=1.1))
    ax.annotate(f'{p_iv_f:.2f}',
                xy=(xf, p_iv_f), xytext=(xf - max(1, xf//5), p_iv_f - 0.08),
                fontsize=9, color=COLOR_IV, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=COLOR_IV, lw=1.1))

    ax.set_xlabel('Bloque de entrenamiento', fontsize=9, color=GRIS)
    ax.set_ylabel('p(TER corto) en el comportamiento', fontsize=9, color=GRIS)
    ax.set_title('Evolución de la distribución de TERs',
                 fontsize=10, color=GRIS, fontweight='bold', pad=4)
    ax.legend(fontsize=9, framealpha=0.9, edgecolor=GRIS_MED)
    ax.set_ylim(0, 1)
    ax.set_xlim(0, len(x) - 1)
    _estilo_ax(ax)


def _panel_desplazamiento(ax, snaps, color, titulo, bins, threshold):
    """
    Panel inferior derecho: distribución de TERs en 4 bloques seleccionados.
    El desplazamiento de la distribución muestra el efecto del entrenamiento.
    """
    epochs  = sorted(snaps.keys())
    alphas  = np.linspace(0.28, 0.92, len(epochs))
    n_total = max(ep for ep in epochs) if epochs else 1

    for i, ep in enumerate(epochs):
        irts = snaps[ep]
        label = (f'Bloque {ep+1}' if ep == epochs[0] else
                 f'Bloque {ep+1}'  if ep == epochs[-1] else
                 f'Bloque {ep+1}')
        ax.hist(np.clip(irts, bins[0], bins[-1]),
                bins=bins, density=True,
                color=color, alpha=float(alphas[i]),
                label=label, histtype='stepfilled')

    ax.axvline(threshold, color='#2d2d2d', linewidth=1.1,
               linestyle='--', alpha=0.45)
    ax.set_xscale('log')
    ax.set_xlabel('TER (s, escala log)', fontsize=9, color=GRIS)
    ax.set_ylabel('Densidad relativa', fontsize=9, color=GRIS)
    ax.set_title(titulo, fontsize=10, color=color, fontweight='bold', pad=4)
    ax.legend(fontsize=7.5, framealpha=0.9, edgecolor=GRIS_MED)
    _estilo_ax(ax)


def hacer_figura(irt_corto, irt_largo, tam_rafaga,
                  rv_n, iv_t, alpha, n_epochs, seed):
    """
    Genera la figura completa con 4 paneles en 3 filas:
      Fila 0 (L|R): Histogramas todo+reforzado para RV e IV
      Fila 1 (completo): Probabilidad relativa de reforzamiento
      Fila 2 (L|R): Evolución p(TER corto) | Distribución en 4 épocas
    """
    # ── Simulación estática (mismo stream para RV e IV) ───────────────────
    N = 10_000
    irts_all, tiempos, _ = generar_irts_rafaga(N, irt_corto, irt_largo, tam_rafaga, seed)
    ref_rv = encontrar_reforzados_rv(irts_all, rv_n,   seed + 1)
    ref_iv = encontrar_reforzados_iv(irts_all, tiempos, iv_t, seed + 2)

    threshold = np.sqrt(irt_corto * irt_largo)
    bins      = _bins(irt_corto, irt_largo)

    # Probabilidades condicionales
    centros_rv, prob_rv = prob_relativa_reforzamiento(irts_all, ref_rv, bins)
    centros_iv, prob_iv = prob_relativa_reforzamiento(irts_all, ref_iv, bins)

    # ── Simulación de evolución ───────────────────────────────────────────
    evol = simular_evolucion(irt_corto, irt_largo, tam_rafaga,
                              rv_n, iv_t, alpha, n_epochs,
                              seed_base=seed + 100)
    p0_real = (tam_rafaga - 1) / tam_rafaga if tam_rafaga > 1 else 0.0

    # ── Figura ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(14, 13), facecolor=BLANCO)
    fig.patch.set_facecolor(BLANCO)
    fig.add_artist(plt.Line2D([0.03, 0.97], [0.990, 0.990],
                               transform=fig.transFigure,
                               color=AZUL, linewidth=3.0))

    gs = GridSpec(3, 2, figure=fig,
                  height_ratios=[1.7, 1.0, 1.5],
                  hspace=0.55, wspace=0.30,
                  top=0.94, bottom=0.06, left=0.08, right=0.97)

    ax_rv   = fig.add_subplot(gs[0, 0])
    ax_iv   = fig.add_subplot(gs[0, 1])
    ax_prob = fig.add_subplot(gs[1, :])
    ax_evol = fig.add_subplot(gs[2, 0])
    ax_desp = fig.add_subplot(gs[2, 1])

    # ── Fila 0: Histogramas ───────────────────────────────────────────────
    _panel_histograma(
        ax_rv, irts_all, ref_rv, COLOR_RV,
        f'Razón Variable (RV-{rv_n})',
        threshold, irt_corto, irt_largo, bins)

    _panel_histograma(
        ax_iv, irts_all, ref_iv, COLOR_IV,
        f'Intervalo Variable (IV-{iv_t}s)',
        threshold, irt_corto, irt_largo, bins)

    # ── Fila 1: Probabilidad condicional ──────────────────────────────────
    _panel_prob_cond(ax_prob, centros_rv, prob_rv, centros_iv, prob_iv, threshold)

    # ── Fila 2: Evolución y desplazamiento ────────────────────────────────
    _panel_evolucion(ax_evol, evol, p0_real, rv_n, iv_t)

    # Usar snaps de RV para panel de desplazamiento (mostrar RV y IV superpuestos)
    # Panel de desplazamiento: mostrar ambos programas en el mismo panel
    epochs_rv  = sorted(evol['snaps_rv'].keys())
    epochs_iv  = sorted(evol['snaps_iv'].keys())
    alphas_seq = np.linspace(0.28, 0.92, len(epochs_rv))

    ax_desp.set_xscale('log')
    for i, (ep_rv, ep_iv) in enumerate(zip(epochs_rv, epochs_iv)):
        irts_rv_s = evol['snaps_rv'][ep_rv]
        irts_iv_s = evol['snaps_iv'][ep_iv]
        a = float(alphas_seq[i])
        label_rv = f'RV — bloque {ep_rv+1}' if i in (0, len(epochs_rv)-1) else '_nolegend_'
        label_iv = f'IV — bloque {ep_iv+1}' if i in (0, len(epochs_iv)-1) else '_nolegend_'
        ax_desp.hist(np.clip(irts_rv_s, bins[0], bins[-1]),
                     bins=bins, density=True, color=COLOR_RV,
                     alpha=a, histtype='step', linewidth=1.5, label=label_rv)
        ax_desp.hist(np.clip(irts_iv_s, bins[0], bins[-1]),
                     bins=bins, density=True, color=COLOR_IV,
                     alpha=a, histtype='step', linewidth=1.5, linestyle='--',
                     label=label_iv)

    ax_desp.axvline(threshold, color='#2d2d2d', linewidth=1.1,
                    linestyle='--', alpha=0.45)
    ax_desp.set_xlabel('TER (s, escala log)', fontsize=9, color=GRIS)
    ax_desp.set_ylabel('Densidad relativa', fontsize=9, color=GRIS)
    ax_desp.set_title('Desplazamiento de la distribución de TERs',
                      fontsize=10, color=GRIS, fontweight='bold', pad=4)
    ax_desp.legend(fontsize=7.5, framealpha=0.9, edgecolor=GRIS_MED)
    _estilo_ax(ax_desp)

    # ── Leyenda global ────────────────────────────────────────────────────
    fig.legend(
        handles=[
            mpatches.Patch(color=COLOR_RV, label='Razón Variable (RV)'),
            mpatches.Patch(color=COLOR_IV, label='Intervalo Variable (IV)'),
            mpatches.Patch(color=GRIS,     label='Distribución base (todos los TERs)'),
        ],
        loc='upper center', ncol=3, fontsize=9,
        framealpha=0.9, edgecolor=GRIS_MED,
        bbox_to_anchor=(0.5, 0.976))

    fig.text(0.5, 0.973,
             'Simulador 13.2 · Reforzamiento Diferencial de Tiempos Entre Respuestas',
             ha='center', va='top', fontsize=13, fontweight='bold', color=AZUL)

    plt.show()

print("✓ Funciones de graficación definidas.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# INTERFAZ INTERACTIVA
# ─────────────────────────────────────────────────────────────────────────

def crear_interfaz():
    kw = dict(style={'description_width': '200px'},
              layout=widgets.Layout(width='470px'))

    encabezado = widgets.HTML(
        '<div style="background:#2C5282;color:white;padding:10px 18px;' +
        'border-radius:6px;font-family:Georgia,serif;font-size:14px;' +
        'font-weight:bold;margin-bottom:10px">' +
        '🔬 Simulador 13.2 &nbsp;·&nbsp; ' +
        'Reforzamiento Diferencial de TERs<br>' +
        '<span style="font-size:11px;font-weight:normal">' +
        'Ajusta los parámetros y presiona <b>▶ Simular</b>. ' +
        'Registra tu predicción <em>antes</em> de correr la simulación.' +
        '</span></div>'
    )

    # ── Comportamiento base ───────────────────────────────────────────────
    sl_irt_c  = widgets.FloatSlider(
        value=0.3, min=0.05, max=1.5, step=0.05,
        description='TER dentro de ráfaga (s):',
        readout_format='.2f', **kw)
    sl_irt_l  = widgets.FloatSlider(
        value=8.0, min=2.0, max=40.0, step=1.0,
        description='Pausa entre ráfagas (s):',
        readout_format='.1f', **kw)
    sl_tam    = widgets.IntSlider(
        value=5, min=2, max=15, step=1,
        description='Tamaño medio de ráfaga:', **kw)

    # ── Programas ─────────────────────────────────────────────────────────
    sl_rv_n   = widgets.IntSlider(
        value=15, min=5, max=60, step=5,
        description='RV — requisito medio (n):', **kw)
    sl_iv_t   = widgets.IntSlider(
        value=60, min=10, max=240, step=10,
        description='IV — intervalo medio (s):', **kw)

    # ── Evolución ─────────────────────────────────────────────────────────
    sl_alpha  = widgets.FloatSlider(
        value=0.15, min=0.03, max=0.50, step=0.02,
        description='Tasa de aprendizaje α:',
        readout_format='.2f', **kw)
    sl_epochs = widgets.IntSlider(
        value=20, min=5, max=60, step=5,
        description='Bloques de entrenamiento:', **kw)
    sl_seed   = widgets.IntSlider(
        value=42, min=1, max=200, step=1,
        description='Semilla aleatoria:', **kw)

    boton = widgets.Button(
        description='▶  Simular',
        layout=widgets.Layout(width='175px', height='40px'),
        style={'button_color': '#2C5282', 'font_weight': 'bold'})

    salida = widgets.Output()

    # ── Etiqueta con umbral calculado ─────────────────────────────────────
    umbral_label = widgets.HTML(
        f'<span style="color:#718096;font-family:Georgia;font-size:11px">' +
        f'Umbral TER corto/largo (√(TERc·TERl)): ' +
        f'<b>{(sl_irt_c.value * sl_irt_l.value)**0.5:.2f} s</b></span>'
    )

    def _actualizar_umbral(*_):
        v = (sl_irt_c.value * sl_irt_l.value) ** 0.5
        umbral_label.value = (
            '<span style="color:#718096;font-family:Georgia;font-size:11px">' +
            f'Umbral TER corto/largo (√(TERc·TERl)): <b>{v:.2f} s</b></span>')

    sl_irt_c.observe(_actualizar_umbral, names='value')
    sl_irt_l.observe(_actualizar_umbral, names='value')

    # ── Layout ────────────────────────────────────────────────────────────
    sep = lambda txt, color: widgets.HTML(
        f'<b style="color:{color};font-family:Georgia;font-size:12px">{txt}</b>')

    col_izq = widgets.VBox([
        sep('Estructura de ráfaga', AZUL),
        sl_irt_c, sl_irt_l, sl_tam, umbral_label,
        widgets.HTML('<br>'),
        sep('Programas de refuerzo', GRIS),
        sl_rv_n, sl_iv_t,
    ], layout=widgets.Layout(padding='8px 16px'))

    col_der = widgets.VBox([
        sep('Parámetros de evolución', GRIS),
        sl_alpha, sl_epochs, sl_seed,
        widgets.HTML('<br>'),
        boton,
    ], layout=widgets.Layout(padding='8px 16px'))

    controles = widgets.HBox([col_izq, col_der])

    def _run(_):
        with salida:
            clear_output(wait=True)
            hacer_figura(
                irt_corto  = sl_irt_c.value,
                irt_largo  = sl_irt_l.value,
                tam_rafaga = sl_tam.value,
                rv_n       = sl_rv_n.value,
                iv_t       = sl_iv_t.value,
                alpha      = sl_alpha.value,
                n_epochs   = sl_epochs.value,
                seed       = sl_seed.value,
            )

    boton.on_click(_run)
    display(encabezado, controles, salida)
    _run(None)

crear_interfaz()


---

## Ejercicios

Registra tu predicción **antes** de cambiar los parámetros. Luego verifica.

---

### Ejercicio 1 · Básico — La asimetría fundamental

Configura los valores predeterminados (TER corto = 0.3 s, pausa = 8 s, ráfaga media = 5, RV-15, IV-60s).

**a)** Observa el panel de probabilidad relativa de reforzamiento (fila central). ¿La curva de RV es plana o tiene pendiente? ¿Qué significa una curva plana en términos de selección?

**b)** ¿La curva de IV sube o baja para TERs largos? ¿Cómo interpretas un valor de 3.0 en ese eje vertical?

**c)** En los paneles superiores, anota el porcentaje de TERs reforzados que son cortos bajo RV y bajo IV. ¿Cuánto difieren del porcentaje base de TERs cortos en la distribución general?

> *Clave:* El porcentaje base de TERs cortos con ráfaga media = 5 es aproximadamente (5−1)/5 = **80%**. ¿Los programas se alejan de ese valor? ¿En qué dirección y cuánto?

---

### Ejercicio 2 · Intermedio — Efecto del tamaño de ráfaga

Mantén RV-15 e IV-60s fijos. Prueba **ráfaga = 2**, **ráfaga = 5** y **ráfaga = 10**.

**a)** ¿Cómo cambia la distribución base de TERs (el histograma gris) cuando aumenta el tamaño de ráfaga? ¿Se vuelve más o menos bimodal?

**b)** ¿El porcentaje de TERs cortos reforzados bajo RV aumenta, disminuye o permanece igual cuando el tamaño de ráfaga aumenta? ¿Y bajo IV?

**c)** Con ráfaga = 2 (casi sin ráfagas, respuestas casi uniformemente distribuidas), ¿cómo cambia la diferencia entre RV e IV? ¿A qué se debe?

---

### Ejercicio 3 · Intermedio — El intervalo del IV

Mantén TER corto = 0.3 s, pausa = 8 s, ráfaga = 5, RV-15 fijos. Prueba IV con **20 s**, **60 s** y **180 s**.

**a)** Con IV-20s: ¿el reforzador llega más durante una ráfaga o durante una pausa? ¿Qué le pasa al porcentaje de TERs cortos reforzados?

**b)** Con IV-180s: el intervalo es mucho mayor que la pausa media (8 s). ¿El reforzador llega más frecuentemente al inicio de una ráfaga nueva o en otros momentos? ¿Por qué?

**c)** ¿Existe un valor del intervalo IV en que el porcentaje de TERs cortos reforzados se iguala al de RV? Predice si ese punto existe y qué pasaría con la evolución del comportamiento en ese caso.

---

### Ejercicio 4 · Avanzado — La evolución

Configura α = 0.15 y 25 bloques de entrenamiento. Observa el panel inferior izquierdo.

**a)** ¿RV o IV produce el cambio más rápido en la distribución de TERs? ¿A qué se debe esa diferencia en velocidad?

**b)** Aumenta α a 0.40. ¿El sistema se vuelve inestable u oscilante? ¿Por qué una tasa de aprendizaje muy alta puede ser contraproducente?

**c)** En el panel de desplazamiento (inferior derecho), describe el movimiento de las distribuciones de RV e IV a lo largo del entrenamiento. ¿Hacia dónde se desplaza cada una? ¿Qué predicción tiene esto para la tasa de respuesta observable en el registro acumulativo?

---

### Ejercicio 5 · Reflexión — Conexión con los datos empíricos

El capítulo describe la evidencia de Catania (1971): con la misma tasa de reforzamiento igualada, los organismos bajo RV responden 5 veces más rápido que bajo IV.

**a)** ¿El mecanismo mostrado en este simulador (reforzamiento diferencial de TERs) predice esa diferencia? ¿Por qué sí o por qué no?

**b)** El capítulo también menciona los programas DRL (*differential reinforcement of low rates*), que exigen que el TER exceda un mínimo para obtener el reforzador. ¿Dónde esperarías que cayera la curva de probabilidad relativa de reforzamiento para un DRL-20s? Dibuja mentalmente la forma de esa curva.

**c)** ¿Qué diferencia conceptual hay entre el efecto del reforzamiento diferencial de TERs (mecanismo molecular, mostrado aquí) y el efecto de la función de retroalimentación (mecanismo molar, Simulador 13.1)? ¿Son mecanismos alternativos o complementarios?


---

## Conexión teórica

Este simulador ilustra el **mecanismo molecular** del reforzamiento diferencial de TERs, que
el capítulo presenta como primera explicación de la diferencia RV vs. IV.

**Los tres paneles del simulador corresponden a tres niveles de análisis:**

**1. Histogramas (fila superior) — descripción:**  
Muestran qué ocurre: la distribución de TERs reforzados difiere de la distribución base bajo IV
pero no bajo RV. Los programas no modifican la frecuencia con que el organismo emite TERs —
eso es un efecto que ocurre *después*, a lo largo del entrenamiento. Lo que difieren inmediatamente
es la distribución de TERs que reciben el consecuente.

**2. Probabilidad relativa (fila media) — mecanismo:**  
Muestra *por qué* ocurre la asimetría. La curva de RV es aproximadamente plana (la probabilidad
de ser reforzado es independiente del TER — el reforzador "cae" en cualquier respuesta con igual
probabilidad relativa). La curva de IV es creciente (los TERs largos tienen *mayor probabilidad
de ser reforzados que los cortos*, a igualdad de frecuencia en la distribución base). Esta es la
diferencia esencial entre los dos programas como mecanismos de selección.

**3. Evolución (fila inferior) — consecuencia:**  
Muestra lo que predice el modelo si los organismos aprenden de los consecuentes: la distribución
de TERs se desplaza hacia los TERs que han sido reforzados. Bajo RV, las ráfagas se prolongan.
Bajo IV, las ráfagas se acortan. El resultado es la diferencia observable en tasas de respuesta
que Catania (1971) demostró con el experimento de cajas acopladas.

---

> **Nota metodológica:** El modelo de aprendizaje usado en la evolución (actualización del tamaño
> medio de ráfaga como función alpha-ponderada del porcentaje de TERs cortos reforzados) es una
> simplificación pedagogica. Modelos más detallados (Killeen, Baum) especifican el mecanismo de
> acoplamiento temporal con mayor precisión. La intuición central — que lo que se refuerza
> determina la forma de la distribución de TERs — es la misma en todos los modelos.
